# Scoring Notebook: Hooks, Defaults, Plugins, Extensions, and ScoreDict

This notebook mirrors the Score API narrative with runnable examples.

It covers:

- scoring hooks and runtime semantics,
- built-in default scorer profiles,
- optional plugin scorers,
- extension patterns for custom scorers,
- each ScorerDictConfig composes ScorerConfig objects and emits a ScoreDict,
- ScoreDict contract and callable persistence lifecycle.

In [1]:
from pathlib import Path
import importlib.util
import json

import numpy as np

from deckard.artifacts import ArtifactLoaderConfig, ScoreDict
from deckard.file import FileConfig
from deckard.score import (
    DefaultClassifierScorerDictConfig,
    DefaultRegressorScorerDictConfig,
    DefaultDataClassificationScorerDictConfig,
    DefaultDataRegressionScorerDictConfig,
    DefaultEvasionAttackScorerDictConfig,
    ScorerConfig,
    ScorerDictConfig,
)

ROOT = Path('.').resolve()
BUILD = ROOT / 'build' / 'notebook_scoring'
BUILD.mkdir(parents=True, exist_ok=True)

print('Notebook root:', ROOT)
print('Build dir:', BUILD)

/Users/c.meyers/.pyenv/versions/3.10.20/envs/deckard/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook root: /Users/c.meyers/Documents/deckard/docs/notebooks
Build dir: /Users/c.meyers/Documents/deckard/docs/notebooks/build/notebook_scoring


## 1) Hooks and Runtime Semantics

Scoring hooks are stage-aware and split-aware. In runtime configs, the key idea is:

- mode controls scope (train/test/val/all, plus runtime-specific scopes),
- stage controls lifecycle boundaries (for example pre-sample, post-pipeline, post-defense),
- plugin hook dispatch runs before/after score stages and merges hook outputs into score_dict.

This means you can add policy behavior in plugins without replacing core scorer execution.

## 2) Inspect Built-in Default Scorer Profiles

In [2]:
defaults = {
    'model_classifier': DefaultClassifierScorerDictConfig(),
    'model_regressor': DefaultRegressorScorerDictConfig(),
    'data_classifier': DefaultDataClassificationScorerDictConfig(),
    'data_regressor': DefaultDataRegressionScorerDictConfig(),
    'attack_evasion': DefaultEvasionAttackScorerDictConfig(),
}

for name, cfg in defaults.items():
    print(name, '->', sorted(cfg.scorers.keys())[:8])

model_classifier -> ['accuracy', 'f1', 'log_loss', 'precision', 'recall', 'roc_auc']
model_regressor -> ['mae', 'mse', 'r2']
data_classifier -> ['class_count_max', 'class_count_min', 'class_imbalance_ratio', 'mutual_information_max', 'mutual_information_mean', 'num_classes']
data_regressor -> ['empirical_cdf', 'mutual_information_max', 'mutual_information_mean']
attack_evasion -> ['accuracy', 'f1-score', 'precision', 'recall', 'success']


## 3) Optional Plugin Scorer Availability

Some scorer defaults are gated by optional dependencies.

In [3]:
optional_modules = ['fairlearn', 'pycanon', 'lifelines']
availability = {m: importlib.util.find_spec(m) is not None for m in optional_modules}
availability

{'fairlearn': True, 'pycanon': True, 'lifelines': True}

## 4) Extension Pattern: Custom ScorerDictConfig

In [4]:
custom = ScorerDictConfig(
    scorers={
        'accuracy': ScorerConfig(score_name='accuracy', score_function='sklearn.metrics.accuracy_score'),
        'f1_weighted': ScorerConfig(
            score_name='f1_weighted',
            score_function='sklearn.metrics.f1_score',
            score_params={'average': 'weighted', 'zero_division': 0},
        ),
    }
)

sorted(custom.scorers.keys())

['accuracy', 'f1_weighted']

## 5) ScoreDict Contract: Scalars, Vectors, Nested Payloads

In [5]:
raw_scores = {
    'accuracy': np.float64(0.91),
    'thresholds': np.array([0.2, 0.5, 0.8]),
    'post-pipeline': {'k_anonymity': 2.0, 'l_diversity': 1.5},
}

scores = ScoreDict.from_payload(raw_scores)
scores

{'accuracy': 0.91,
 'thresholds': [0.2, 0.5, 0.8],
 'post-pipeline': {'k_anonymity': 2.0, 'l_diversity': 1.5}}

## 6) Update Scores by Stage, Mode, and Split

In [6]:
scores.update_score(0.93, key='f1', stage='post-pipeline', mode='test', split='fold-0')
scores.update_score({'precision': 0.90, 'recall': 0.88}, stage='post-pipeline', mode='test')
scores.update_score(0.12, key='demographic_parity_difference', stage='post-defense', mode='test')

scores

{'accuracy': 0.91,
 'thresholds': [0.2, 0.5, 0.8],
 'post-pipeline': {'k_anonymity': 2.0,
  'l_diversity': 1.5,
  'test': {'fold-0': {'f1': 0.93}, 'precision': 0.9, 'recall': 0.88}},
 'post-defense': {'test': {'demographic_parity_difference': 0.12}}}

## 7) Inspect Flat and Dotlist Projections

In [7]:
flat = scores.flatten()
flat_by_scope = scores.flat_by_scope()
dotlist_dict = scores.dotlist_dict()
dotlist_items = scores.dotlist_items()

print('Flat keys sample:', list(flat.keys())[:8])
print('Scope keys sample:', list(flat_by_scope.keys())[:8])
print('Dotlist sample:', dotlist_items[:5])

Flat keys sample: ['accuracy', 'thresholds', 'post-pipeline.k_anonymity', 'post-pipeline.l_diversity', 'post-pipeline.test.fold-0.f1', 'post-pipeline.test.precision', 'post-pipeline.test.recall', 'post-defense.test.demographic_parity_difference']
Scope keys sample: ['accuracy', 'thresholds', 'post-pipeline', 'post-defense']
Dotlist sample: ['accuracy=0.91', 'thresholds=[0.2, 0.5, 0.8]', 'post-pipeline.k_anonymity=2.0', 'post-pipeline.l_diversity=1.5', 'post-pipeline.test.fold-0.f1=0.93']


## 8) Callable Persistence Lifecycle (Load/Merge/Save)

When a score file is provided, calling ScoreDict performs persistence through ArtifactLoaderConfig.

In [8]:
file_cfg = FileConfig(score_file=str(BUILD / 'scores.json'))
artifact_loader = ArtifactLoaderConfig(payload_kind='score')

first_run = ScoreDict.from_payload({'accuracy': 0.75, 'post-pipeline': {'k_anonymity': 2}})
resolved_first = first_run(score_file=file_cfg.score_file, artifact_loader=artifact_loader, persist=True)

second_run = ScoreDict.from_payload({'accuracy': 0.82, 'post-defense': {'fairness_gap': 0.07}})
resolved_second = second_run(score_file=file_cfg.score_file, artifact_loader=artifact_loader, persist=True)

print('First resolved payload:', resolved_first)
print('Second resolved payload:', resolved_second)

First resolved payload: {'accuracy': 0.75, 'post-pipeline': {'k_anonymity': 2}, 'post-defense': {'fairness_gap': 0.07}}
Second resolved payload: {'accuracy': 0.75, 'post-defense': {'fairness_gap': 0.07}, 'post-pipeline': {'k_anonymity': 2}}


## 9) Verify Persisted Contract Envelope

In [9]:
with open(file_cfg.score_file, 'r', encoding='utf-8') as handle:
    persisted = json.load(handle)

required_keys = ['_schema', 'payload', 'flat', 'flat_by_scope', 'dotlist', 'dotlist_items']
print({k: k in persisted for k in required_keys})
print('schema:', persisted.get('_schema'))
print('payload keys:', list(persisted.get('payload', {}).keys()))

{'_schema': True, 'payload': True, 'flat': True, 'flat_by_scope': True, 'dotlist': True, 'dotlist_items': True}
schema: deckard.score.v1
payload keys: ['accuracy', 'post-defense', 'post-pipeline']


## 10) Runtime-only Path (No Score File)

Without a score file, ScoreDict returns the nested runtime payload and does not persist.

In [10]:
ephemeral = ScoreDict.from_payload({'train': {'loss': 0.31}, 'test': {'loss': 0.28}})
ephemeral()

{'train': {'loss': 0.31}, 'test': {'loss': 0.28}}